## 1. Khởi tạo môi trường và Tải dữ liệu
Trong phần này, chúng ta sẽ nạp các thư viện cần thiết và tiến hành đọc dữ liệu điểm thi THPT Quốc gia từ năm 2020 đến 2024. Dữ liệu sẽ được gộp chung vào một DataFrame duy nhất (`df_raw`), đi kèm một cột `year` để phân biệt năm thi, giúp dễ dàng duyệt qua và kiểm tra chất lượng tự động.

In [ ]:
import pandas as pd
import numpy as np

# Danh sách các năm cần phân tích
years = [2020, 2021, 2022, 2023, 2024]
frames = []

print("Đang tải dữ liệu...")
for year in years:
    # Ép kiểu sbd về string để giữ nguyên các số 0 ở đầu
    df_temp = pd.read_csv(f'raw/thpt{year}.csv', dtype={'sbd': str})
    df_temp['year'] = year
    frames.append(df_temp)
    print(f" - Đã nạp thành công năm {year}: {len(df_temp):,} bản ghi")

# Gộp toàn bộ dữ liệu thành một DataFrame
if frames:
    df_raw = pd.concat(frames, ignore_index=True)
    print(f"\nTổng số bản ghi toàn hệ thống: {len(df_raw):,}")

Đang tải dữ liệu...
 - Đã nạp thành công năm 2020: 870,486 bản ghi
 - Đã nạp thành công năm 2021: 914,558 bản ghi
 - Đã nạp thành công năm 2022: 995,435 bản ghi
 - Đã nạp thành công năm 2023: 1,017,584 bản ghi
 - Đã nạp thành công năm 2024: 1,061,604 bản ghi

Tổng số bản ghi toàn hệ thống: 4,859,667


## 2. Kiểm tra dữ liệu trùng lặp theo từng năm
Kiểm tra tính duy nhất của dữ liệu là bước bắt buộc. Một bản ghi có thể bị trùng lặp hoàn toàn (tất cả các cột giống nhau) hoặc nghiêm trọng hơn là trùng lặp khóa chính - Số báo danh (`sbd`) nhưng điểm số lại khác nhau (do lỗi cào dữ liệu).

In [54]:
print("BÁO CÁO KIỂM TRA TRÙNG LẶP DỮ LIỆU\n" + "="*40)

for year in years:
    df_year = df_raw[df_raw['year'] == year]
    
    # 1. Trùng lặp hoàn toàn 100% các cột
    full_dupes = df_year.duplicated().sum()
    
    # 2. Trùng lặp Số báo danh (SBD)
    sbd_dupes = df_year.duplicated(subset=['sbd']).sum()
    
    print(f"NĂM {year} (Tổng: {len(df_year):,} thí sinh):")
    print(f" - Trùng lặp toàn bộ dòng: {full_dupes} bản ghi")
    print(f" - Trùng lặp Số báo danh:  {sbd_dupes} bản ghi")
    
    if sbd_dupes > 0:
        print("   CẢNH BÁO: Phát hiện SBD trùng lặp!")
    else:
        print("   Dữ liệu SBD định danh duy nhất, không có trùng lặp.")
    print("-" * 40)

BÁO CÁO KIỂM TRA TRÙNG LẶP DỮ LIỆU
NĂM 2020 (Tổng: 870,486 thí sinh):
 - Trùng lặp toàn bộ dòng: 0 bản ghi
 - Trùng lặp Số báo danh:  0 bản ghi
   Dữ liệu SBD định danh duy nhất, không có trùng lặp.
----------------------------------------
NĂM 2021 (Tổng: 914,558 thí sinh):
 - Trùng lặp toàn bộ dòng: 0 bản ghi
 - Trùng lặp Số báo danh:  0 bản ghi
   Dữ liệu SBD định danh duy nhất, không có trùng lặp.
----------------------------------------
NĂM 2022 (Tổng: 995,435 thí sinh):
 - Trùng lặp toàn bộ dòng: 0 bản ghi
 - Trùng lặp Số báo danh:  0 bản ghi
   Dữ liệu SBD định danh duy nhất, không có trùng lặp.
----------------------------------------
NĂM 2023 (Tổng: 1,017,584 thí sinh):
 - Trùng lặp toàn bộ dòng: 0 bản ghi
 - Trùng lặp Số báo danh:  0 bản ghi
   Dữ liệu SBD định danh duy nhất, không có trùng lặp.
----------------------------------------
NĂM 2024 (Tổng: 1,061,604 thí sinh):
 - Trùng lặp toàn bộ dòng: 0 bản ghi
 - Trùng lặp Số báo danh:  0 bản ghi
   Dữ liệu SBD định danh duy nhấ

## 3. Kiểm tra giá trị khuyết thiếu (Missing Values) theo từng năm
Việc thống kê `NaN` giúp chúng ta nhận diện tỷ lệ thí sinh bỏ thi hoặc không đăng ký thi ở từng môn học. Điểm `NaN` trong ngữ cảnh này mang ý nghĩa định lượng (không thi) chứ không phải là lỗi dữ liệu cần điền khuyết (Imputation).

In [55]:
# Khai báo các cột môn học
SUBJECTS = ['toan', 'ngu_van', 'ngoai_ngu', 'vat_ly', 'hoa_hoc', 'sinh_hoc', 'lich_su', 'dia_ly', 'gdcd']

print("BÁO CÁO TỶ LỆ DỮ LIỆU KHUYẾT THIẾU (THÍ SINH KHÔNG THI)\n" + "="*60)

for year in years:
    df_year = df_raw[df_raw['year'] == year]
    total_students = len(df_year)
    
    print(f"\nNĂM {year} (Tổng: {total_students:,} thí sinh)")
    
    # Đếm số lượng giá trị NaN cho từng môn
    missing_counts = df_year[SUBJECTS].isna().sum()
    
    # Chỉ giữ lại các môn có giá trị thiếu và sắp xếp giảm dần
    missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)
    
    if missing_counts.empty:
        print(" - Không có giá trị khuyết thiếu nào.")
    else:
        # Tạo bảng thống kê có kèm tỷ lệ phần trăm
        missing_df = pd.DataFrame({
            'Số lượng NaN': missing_counts,
            'Tỷ lệ (%)': (missing_counts / total_students) * 100
        })
        
        # Format lại tỷ lệ phần trăm để dễ đọc
        missing_df['Tỷ lệ (%)'] = missing_df['Tỷ lệ (%)'].apply(lambda x: f"{x:.2f}%")
        print(missing_df.to_string())

BÁO CÁO TỶ LỆ DỮ LIỆU KHUYẾT THIẾU (THÍ SINH KHÔNG THI)

NĂM 2020 (Tổng: 870,486 thí sinh)
           Số lượng NaN Tỷ lệ (%)
sinh_hoc         580109    66.64%
vat_ly           577199    66.31%
hoa_hoc          574950    66.05%
gdcd             387506    44.52%
dia_ly           315414    36.23%
lich_su          301905    34.68%
ngoai_ngu         98388    11.30%
ngu_van           13921     1.60%
toan               3905     0.45%

NĂM 2021 (Tổng: 914,558 thí sinh)
           Số lượng NaN Tỷ lệ (%)
sinh_hoc         593271    64.87%
vat_ly           589720    64.48%
hoa_hoc          588172    64.31%
gdcd             426364    46.62%
dia_ly           337680    36.92%
lich_su          332341    36.34%
ngoai_ngu        110273    12.06%
ngu_van           12568     1.37%
toan               9292     1.02%

NĂM 2022 (Tổng: 995,435 thí sinh)
           Số lượng NaN Tỷ lệ (%)
sinh_hoc         673237    67.63%
vat_ly           669912    67.30%
hoa_hoc          668068    67.11%
gdcd             441092

## 4. Kiểm tra các dòng dữ liệu khuyết hoàn toàn
Trong bước này, chúng ta xác định những bản ghi mà tất cả các cột điểm môn học đều là `NaN`. Những thí sinh này có thể đã bỏ thi toàn bộ các môn hoặc đây là các dòng trống phát sinh trong quá trình thu thập dữ liệu. Việc loại bỏ chúng giúp làm sạch tập mẫu, đảm bảo các phân tích về tỷ lệ đỗ/trượt và điểm trung bình được chính xác.

In [56]:
# Danh sách 9 môn học dùng để kiểm tra khuyết thiếu
SUBJECTS = ['toan', 'ngu_van', 'ngoai_ngu', 'vat_ly', 'hoa_hoc', 'sinh_hoc', 'lich_su', 'dia_ly', 'gdcd']

print("BÁO CÁO THỐNG KÊ DÒNG KHUYẾT TOÀN PHẦN THEO NĂM\n" + "="*55)

# Biến để tính tổng số dòng trống trên toàn bộ các năm
total_empty_all_years = 0

for year in sorted(df_raw['year'].unique()):
    # Lọc dữ liệu theo năm để báo cáo chi tiết
    df_year = df_raw[df_raw['year'] == year]
    
    # Kiểm tra các dòng mà tất cả các môn trong SUBJECTS đều là NaN
    is_empty = df_year[SUBJECTS].isna().all(axis=1)
    num_empty = is_empty.sum()
    total_empty_all_years += num_empty
    
    print(f"NĂM {year}:")
    print(f" - Số dòng trống hoàn toàn: {num_empty:,} / {len(df_year):,} bản ghi")
    print(f" - Tỷ lệ: {(num_empty/len(df_year))*100:.4f}%")
    
    if num_empty > 0:
        print(f"   => Phát hiện {num_empty:,} bản ghi rác tiềm năng.")
    else:
        print("   => Dữ liệu sạch, không có bản ghi trống.")
    print("-" * 40)

print(f"\nTỔNG KẾT: Phát hiện {total_empty_all_years:,} dòng trống trên tổng số {len(df_raw):,} bản ghi.")

BÁO CÁO THỐNG KÊ DÒNG KHUYẾT TOÀN PHẦN THEO NĂM
NĂM 2020:
 - Số dòng trống hoàn toàn: 0 / 870,486 bản ghi
 - Tỷ lệ: 0.0000%
   => Dữ liệu sạch, không có bản ghi trống.
----------------------------------------
NĂM 2021:
 - Số dòng trống hoàn toàn: 0 / 914,558 bản ghi
 - Tỷ lệ: 0.0000%
   => Dữ liệu sạch, không có bản ghi trống.
----------------------------------------
NĂM 2022:
 - Số dòng trống hoàn toàn: 0 / 995,435 bản ghi
 - Tỷ lệ: 0.0000%
   => Dữ liệu sạch, không có bản ghi trống.
----------------------------------------
NĂM 2023:
 - Số dòng trống hoàn toàn: 0 / 1,017,584 bản ghi
 - Tỷ lệ: 0.0000%
   => Dữ liệu sạch, không có bản ghi trống.
----------------------------------------
NĂM 2024:
 - Số dòng trống hoàn toàn: 0 / 1,061,604 bản ghi
 - Tỷ lệ: 0.0000%
   => Dữ liệu sạch, không có bản ghi trống.
----------------------------------------

TỔNG KẾT: Phát hiện 0 dòng trống trên tổng số 4,859,667 bản ghi.


## 5. Kiểm tra thí sinh dự thi không đủ 3 môn
Thống kê nhóm thí sinh có tổng số môn thi ít hơn 3. Việc nhận diện nhóm này giúp phân loại được thí sinh tự do thi bổ sung hoặc các trường hợp thí sinh bỏ thi giữa chừng. Ta sẽ tạo thêm một cột phụ `num_subjects` để lưu số lượng môn mà mỗi thí sinh thực tế có điểm.

In [57]:
# 1. Tính tổng số môn thi mỗi thí sinh tham gia (số cột không phải NaN)
df_raw['num_subjects'] = df_raw[SUBJECTS].notna().sum(axis=1)

print("BÁO CÁO THÍ SINH THI ÍT HƠN 3 MÔN THEO NĂM\n" + "="*50)

for year in sorted(df_raw['year'].unique()):
    df_year = df_raw[df_raw['year'] == year]
    
    # Lọc thí sinh thi < 3 môn
    less_than_3 = df_year[df_year['num_subjects'] < 3]
    num_less_than_3 = len(less_than_3)
    
    print(f"NĂM {year}:")
    print(f" - Số thí sinh thi < 3 môn: {num_less_than_3:,} / {len(df_year):,} bản ghi")
    print(f" - Tỷ lệ: {(num_less_than_3/len(df_year))*100:.2f}%")
    
    if num_less_than_3 > 0:
        # Hiển thị phân bổ chi tiết: thi 1 môn và 2 môn
        dist = less_than_3['num_subjects'].value_counts().sort_index()
        for count, total in dist.items():
            print(f"   + Thi {count} môn: {total:,} thí sinh")
    print("-" * 40)

BÁO CÁO THÍ SINH THI ÍT HƠN 3 MÔN THEO NĂM
NĂM 2020:
 - Số thí sinh thi < 3 môn: 912 / 870,486 bản ghi
 - Tỷ lệ: 0.10%
   + Thi 1 môn: 323 thí sinh
   + Thi 2 môn: 589 thí sinh
----------------------------------------
NĂM 2021:
 - Số thí sinh thi < 3 môn: 836 / 914,558 bản ghi
 - Tỷ lệ: 0.09%
   + Thi 1 môn: 371 thí sinh
   + Thi 2 môn: 465 thí sinh
----------------------------------------
NĂM 2022:
 - Số thí sinh thi < 3 môn: 1,050 / 995,435 bản ghi
 - Tỷ lệ: 0.11%
   + Thi 1 môn: 414 thí sinh
   + Thi 2 môn: 636 thí sinh
----------------------------------------
NĂM 2023:
 - Số thí sinh thi < 3 môn: 1,063 / 1,017,584 bản ghi
 - Tỷ lệ: 0.10%
   + Thi 1 môn: 453 thí sinh
   + Thi 2 môn: 610 thí sinh
----------------------------------------
NĂM 2024:
 - Số thí sinh thi < 3 môn: 1,591 / 1,061,604 bản ghi
 - Tỷ lệ: 0.15%
   + Thi 1 môn: 729 thí sinh
   + Thi 2 môn: 862 thí sinh
----------------------------------------


## 5. Kiểm tra tính hợp lệ của giá trị điểm (Outliers & Range Check)
Sau khi loại bỏ dòng trống, ta cần đảm bảo tất cả điểm số hiện có phải nằm trong khoảng hợp lệ từ $0$ đến $10$. Mọi giá trị nằm ngoài khoảng này hoặc có định dạng lạ (không phải số) sẽ được ghi nhận là lỗi dữ liệu.

In [58]:
# Danh sách các chỉ số trong thống kê 5 số
stats_labels = ['min', '25%', '50%', '75%', 'max']

print("THỐNG KÊ 5 SỐ CHO CÁC MÔN THI THEO TỪNG NĂM\n" + "="*60)

# Duyệt qua từng năm để tính toán
for year in sorted(df_raw['year'].unique()):
    print(f"\nNĂM THI: {year}")
    
    # Lấy dữ liệu của năm hiện tại và tính toán describe
    # Ta chọn các dòng tương ứng với 5 số thống kê
    year_summary = df_raw[df_raw['year'] == year][SUBJECTS].describe().loc[stats_labels]
    
    year_summary.index = ['Nhỏ nhất (Min)', 'Tứ phân vị Q1 (25%)', 'Trung vị (Median)', 'Tứ phân vị Q3 (75%)', 'Lớn nhất (Max)']
    
    # Hiển thị bảng kết quả
    display(year_summary)
    print("-" * 60)

THỐNG KÊ 5 SỐ CHO CÁC MÔN THI THEO TỪNG NĂM

NĂM THI: 2020


,toan,ngu_van,ngoai_ngu,vat_ly,hoa_hoc,sinh_hoc,lich_su,dia_ly,gdcd
Nhỏ nhất (Min),0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
Tứ phân vị Q1 (25%),5.40,6.00,3.20,5.75,5.50,4.75,4.00,6.00,7.50
Trung vị (Median),7.00,6.75,4.20,7.00,7.00,5.50,5.00,7.00,8.25
Tứ phân vị Q3 (75%),8.00,7.50,5.60,7.75,8.00,6.50,6.25,7.50,9.00
Lớn nhất (Max),10.00,10.00,10.00,10.00,10.00,10.00,10.00,10.00,10.00


------------------------------------------------------------

NĂM THI: 2021


,toan,ngu_van,ngoai_ngu,vat_ly,hoa_hoc,sinh_hoc,lich_su,dia_ly,gdcd
Nhỏ nhất (Min),0.60,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
Tứ phân vị Q1 (25%),5.60,5.75,4.00,5.75,5.50,4.50,3.50,6.25,7.75
Trung vị (Median),7.00,6.50,5.60,6.75,7.00,5.50,4.75,7.00,8.50
Tứ phân vị Q3 (75%),8.00,7.50,7.80,7.75,7.75,6.50,6.25,7.75,9.25
Lớn nhất (Max),10.00,10.00,10.00,10.00,10.00,10.00,10.00,10.00,10.00


------------------------------------------------------------

NĂM THI: 2022


...

------------------------------------------------------------

NĂM THI: 2023


,toan,ngu_van,ngoai_ngu,vat_ly,hoa_hoc,sinh_hoc,lich_su,dia_ly,gdcd
Nhỏ nhất (Min),0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
Tứ phân vị Q1 (25%),5.20,6.00,4.00,5.50,5.75,5.50,5.00,5.50,7.75
Trung vị (Median),6.60,7.00,5.20,6.75,7.00,6.50,6.00,6.25,8.50
Tứ phân vị Q3 (75%),7.60,7.75,7.00,7.75,7.75,7.25,7.25,7.00,9.25
Lớn nhất (Max),10.00,10.00,10.00,10.00,10.00,10.00,10.00,10.00,10.00


------------------------------------------------------------

NĂM THI: 2024


,toan,ngu_van,ngoai_ngu,vat_ly,hoa_hoc,sinh_hoc,lich_su,dia_ly,gdcd
Nhỏ nhất (Min),0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
Tứ phân vị Q1 (25%),5.40,6.50,4.00,5.50,5.50,5.50,5.50,6.50,7.50
Trung vị (Median),6.80,7.50,5.20,7.00,6.75,6.25,6.50,7.25,8.25
Tứ phân vị Q3 (75%),7.60,8.25,7.00,8.00,8.00,7.25,7.50,8.00,9.00
Lớn nhất (Max),9.80,10.00,10.00,10.00,10.00,10.00,10.00,10.00,10.00


------------------------------------------------------------


## 6. Loại bỏ bản ghi trống toàn phần và Lưu trữ dữ liệu sạch (Data Export)
Sau khi đã kiểm tra và xác định các dòng "rác" (không có điểm ở tất cả 9 môn), chúng ta tiến hành loại bỏ chúng vĩnh viễn khỏi tập dữ liệu. Kết quả cuối cùng sẽ được phân tách theo từng năm và lưu vào thư mục `data/processed` để làm nguồn dữ liệu đầu vào chuẩn cho Dashboard.

In [59]:
import os

# 1. Xác định danh sách các cột cần loại bỏ
COLS_TO_DROP = ['year', 'nam', 'Cum_Thi', 'ma_ngoai_ngu', 'num_subjects']
SUBJECTS_ONLY = ['toan', 'ngu_van', 'ngoai_ngu', 'vat_ly', 'hoa_hoc', 'sinh_hoc', 'lich_su', 'dia_ly', 'gdcd']

# 2. Tạo thư mục 'data/processed' nếu chưa tồn tại
output_dir = "data/processed"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Đã tạo thư mục: {output_dir}")

print("BẮT ĐẦU QUY TRÌNH TINH GỌN VÀ LƯU TRỮ...\n" + "="*50)

for year in sorted(df_raw['year'].unique()):
    # Lọc dữ liệu theo năm
    df_year = df_raw[df_raw['year'] == year].copy()
    initial_count = len(df_year)
    
    # --- BƯỚC 1: LOẠI BỎ DÒNG KHUYẾT TOÀN PHẦN ---
    df_year = df_year.dropna(subset=SUBJECTS_ONLY, how='all')
    removed_rows = initial_count - len(df_year)
    
    # --- BƯỚC 2: LOẠI BỎ CÁC CỘT KHÔNG CẦN THIẾT ---
    # Sử dụng errors='ignore' để không báo lỗi nếu cột đó không tồn tại trong file của năm đó
    df_year = df_year.drop(columns=COLS_TO_DROP, errors='ignore')
    
    # --- BƯỚC 3: LƯU FILE SẠCH ---
    output_path = os.path.join(output_dir, f"thpt{year}.csv")
    df_year.to_csv(output_path, index=False)
    
    print(f"NĂM {year}:")
    print(f" - Đã loại bỏ: {removed_rows:,} dòng trống")
    print(f" - Các cột còn lại: {list(df_year.columns)}")
    print(f" - Đã lưu tại: {output_path}")
    print("-" * 40)

print("\nHOÀN TẤT: Dữ liệu đã được tinh gọn và sẵn sàng cho Dashboard!")

BẮT ĐẦU QUY TRÌNH TINH GỌN VÀ LƯU TRỮ...
NĂM 2020:
 - Đã loại bỏ: 0 dòng trống
 - Các cột còn lại: ['sbd', 'toan', 'ngu_van', 'vat_ly', 'hoa_hoc', 'sinh_hoc', 'lich_su', 'dia_ly', 'gdcd', 'ngoai_ngu', 'ma_mon_ngoai_ngu']
 - Đã lưu tại: data/processed\thpt2020.csv
----------------------------------------
NĂM 2021:
 - Đã loại bỏ: 0 dòng trống
 - Các cột còn lại: ['sbd', 'toan', 'ngu_van', 'vat_ly', 'hoa_hoc', 'sinh_hoc', 'lich_su', 'dia_ly', 'gdcd', 'ngoai_ngu', 'ma_mon_ngoai_ngu']
 - Đã lưu tại: data/processed\thpt2021.csv
----------------------------------------
NĂM 2022:
 - Đã loại bỏ: 0 dòng trống
 - Các cột còn lại: ['sbd', 'toan', 'ngu_van', 'vat_ly', 'hoa_hoc', 'sinh_hoc', 'lich_su', 'dia_ly', 'gdcd', 'ngoai_ngu', 'ma_mon_ngoai_ngu']
 - Đã lưu tại: data/processed\thpt2022.csv
----------------------------------------
NĂM 2023:
 - Đã loại bỏ: 0 dòng trống
 - Các cột còn lại: ['sbd', 'toan', 'ngu_van', 'vat_ly', 'hoa_hoc', 'sinh_hoc', 'lich_su', 'dia_ly', 'gdcd', 'ngoai_ngu', 'ma_mon

## Chuẩn hóa định dạng Số báo danh (SBD)
Trong bước này, chúng ta tiến hành loại bỏ các ký tự tiền tố không cần thiết (ví dụ: mã năm 20 bị dính vào trước SBD trong quá trình thu thập dữ liệu). Mục tiêu là đưa Số báo danh về định dạng gốc bằng cách cắt bỏ 2 ký tự đầu tiên.

In [51]:
import pandas as pd

# Đọc file
df = pd.read_csv('processed/thpt2020.csv', dtype={'sbd': str})

# Bỏ 2 ký tự đầu tiên, giữ lại toàn bộ phần còn lại
df['sbd'] = df['sbd'].str[2:]

# Lưu file
df.to_csv('processed/thpt2020.csv', index=False)

print("Đã cắt 2 ký tự đầu của SBD và lưu file thành công!")

Đã cắt 2 ký tự đầu của SBD và lưu file thành công!
